# LLM Output Analysis
Exploratory comparison of LLM-generated data cleaning outputs against ground truth, across models and tasks.

## Setup
Set `TASK` and `MODELS` below, then run all cells. Put data dir in the .env file (NEXTSTEPS_DATA_DIR=dir)

## Load Data and packages

In [7]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'python-dotenv']:
    try:
        __import__(pkg.replace('-', '_').split('.')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
BASE_DIR = Path(os.environ['NEXTSTEPS_DATA_DIR'])

# Set tasks and models to compare
TASKS  = [18, 1]
MODELS = ['qwen3.5_9b', 'devstral', 'qwen3.5_397']

In [ ]:
# Load ground truth and LLM prediction for each task and model
id_col = 'NSID'
data = {}  # data[task][model]

for task in TASKS:
    data[task] = {}
    gt_cols_ref = None

    for model in MODELS:
        matches = list((BASE_DIR / model).glob(f'sample{task}_*'))
        assert len(matches) == 1, f'Task {task} / {model}: expected 1 match, found {len(matches)}'
        out_dir = matches[0] / 'data' / 'output'

        gt   = pd.read_csv(out_dir / 'output.csv', low_memory=False)
        pred = pd.read_csv(out_dir / 'cleaned_data.csv', low_memory=False)

        if gt_cols_ref is None:
            gt_cols_ref = [c for c in gt.columns if c != id_col]

        shared = [c for c in gt_cols_ref if c in pred.columns]
        merged = gt.merge(pred, on=id_col, suffixes=('_gt', '_pred'))

        data[task][model] = {'merged': merged, 'shared_cols': shared}

    data[task]['_gt_cols'] = gt_cols_ref
    print(f'Task {task}: GT columns -> {gt_cols_ref}')
    for model in MODELS:
        shared = data[task][model]['shared_cols']
        missing = [c for c in gt_cols_ref if c not in shared]
        status = 'OK' if not missing else f'MISSING: {missing}'
        print(f'  {model}: {status}')

## Cross-tabulation Plots
Each plot shows ground truth (rows) vs LLM prediction (cols). Blue = match, orange = mismatch.

In [ ]:
cmap = ListedColormap(['#f4a582', '#2166ac'])  # orange = mismatch, blue = match

for task in TASKS:
    gt_cols = data[task]['_gt_cols']
    print(f'\n{"="*60}\nTask {task}\n{"="*60}')

    for col in gt_cols:
        fig, axes = plt.subplots(1, len(MODELS), figsize=(7 * len(MODELS), 6))
        if len(MODELS) == 1:
            axes = [axes]

        for ax, model in zip(axes, MODELS):
            if col not in data[task][model]['shared_cols']:
                ax.axis('off')
                ax.text(0.5, 0.5,
                        f'Variable not produced\nby {model}',
                        ha='center', va='center', fontsize=11,
                        color='grey', style='italic',
                        transform=ax.transAxes)
                ax.set_title(f'{model}\n{col}', fontsize=11)
                continue

            merged = data[task][model]['merged']
            gt_col, pred_col = f'{col}_gt', f'{col}_pred'

            ct = pd.crosstab(merged[gt_col], merged[pred_col], margins=False)
            diag = pd.DataFrame(0, index=ct.index, columns=ct.columns)
            for val in ct.index:
                if val in ct.columns:
                    diag.loc[val, val] = 1

            sns.heatmap(diag, annot=ct, fmt='d', cmap=cmap, vmin=0, vmax=1,
                        linewidths=0.5, cbar=False, ax=ax)

            match_rate = (merged[gt_col] == merged[pred_col]).mean()
            ax.set_title(f'{model}\n{col}  (match: {match_rate:.1%})', fontsize=11)
            ax.set_xlabel('LLM prediction')
            ax.set_ylabel('Ground truth')

        plt.tight_layout()
        plt.show()